# Work Hours Analysis - CVS Pharmacy Dataset

Analyze working patterns in the original event log to determine:
- What are the working hours?
- Do agents work on weekends?
- Are there night shifts or outliers?
- Do patterns differ per agent?

This informs whether the simulator needs a work schedule.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Load data
df = pd.read_csv("../../data/cvs_pharmacy/processed/cvs_pharmacy.csv")
df['start_timestamp'] = pd.to_datetime(df['start_timestamp'], format='ISO8601')
df['end_timestamp'] = pd.to_datetime(df['end_timestamp'], format='ISO8601')

# Derived columns
df['start_hour'] = df['start_timestamp'].dt.hour
df['end_hour'] = df['end_timestamp'].dt.hour
df['weekday'] = df['start_timestamp'].dt.day_name()
df['weekday_num'] = df['start_timestamp'].dt.dayofweek  # 0=Mon, 6=Sun
df['date'] = df['start_timestamp'].dt.date
df['duration_minutes'] = (df['end_timestamp'] - df['start_timestamp']).dt.total_seconds() / 60

print(f"Total tasks: {len(df):,}")
print(f"Date range: {df['start_timestamp'].min().date()} to {df['start_timestamp'].max().date()}")
print(f"Unique agents: {df['resource'].nunique()}")
print(f"Unique activities: {df['activity_name'].nunique()}")
print(f"\nAgents: {sorted(df['resource'].unique())}")

## 1. Overview: Tasks per Weekday

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_counts = df['weekday'].value_counts().reindex(weekday_order).fillna(0)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#4CAF50' if c > 0 else '#E0E0E0' for c in weekday_counts]
weekday_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Tasks per Weekday', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Tasks')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)

# Annotate
for i, v in enumerate(weekday_counts):
    if v > 0:
        ax.text(i, v + 200, f'{int(v):,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

weekend_tasks = df[df['weekday_num'] >= 5]
print(f"Weekend tasks: {len(weekend_tasks)} ({len(weekend_tasks)/len(df)*100:.1f}%)")

## 2. Start Hour Distribution (All Agents)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

hour_counts = df['start_hour'].value_counts().sort_index()
all_hours = pd.Series(0, index=range(24))
all_hours.update(hour_counts)

colors = ['#4CAF50' if c > 0 else '#E0E0E0' for c in all_hours]
all_hours.plot(kind='bar', ax=ax, color=colors, edgecolor='white', width=0.8)

ax.set_title('Task Start Hour Distribution (All Agents)', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Tasks')
ax.set_xlabel('Hour of Day')
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h:02d}' for h in range(24)])

# Mark working hours
ax.axvspan(7.5, 19.5, alpha=0.1, color='green', label='Working hours (08-19)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Earliest start: {df['start_hour'].min():02d}:00")
print(f"Latest start:   {df['start_hour'].max():02d}:00")
print(f"Tasks before 08:00: {len(df[df['start_hour'] < 8])}")
print(f"Tasks after 19:00:  {len(df[df['start_hour'] > 19])}")

## 3. Start Hour per Agent (Heatmap)

In [ ]:
# Pivot: agent x hour
agent_hour = df.groupby(['resource', 'start_hour']).size().unstack(fill_value=0)

# Ensure all 24 hours are present
for h in range(24):
    if h not in agent_hour.columns:
        agent_hour[h] = 0
agent_hour = agent_hour[sorted(agent_hour.columns)]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    agent_hour, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
    linewidths=0.5, linecolor='white',
    xticklabels=[f'{h:02d}' for h in range(24)],
    cbar_kws={'label': 'Task Count'}
)
ax.set_title('Task Count per Agent per Hour', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

## 3b. First & Last Task Hour per Agent per Day

In [ ]:
# For each agent+date: find earliest start and latest timestamp (start OR end)
agent_day = df.groupby(['resource', 'date']).agg(
    first_time=('start_timestamp', lambda x: x.min().hour + x.min().minute / 60 + x.min().second / 3600),
    last_start=('start_timestamp', lambda x: x.max().hour + x.max().minute / 60 + x.max().second / 3600),
    last_end=('end_timestamp', lambda x: x.max().hour + x.max().minute / 60 + x.max().second / 3600),
    n_tasks=('start_timestamp', 'count'),
).reset_index()

# Use the latest of (last start, last end) as the true end of the workday
agent_day['last_time'] = agent_day[['last_start', 'last_end']].max(axis=1)

agents = sorted(df['resource'].unique())
palette = sns.color_palette('Set2', len(agents))

fig, axes = plt.subplots(len(agents), 1, figsize=(12, 3 * len(agents)), sharex=True)
if len(agents) == 1:
    axes = [axes]

for ax, agent, color in zip(axes, agents, palette):
    ad = agent_day[agent_day['resource'] == agent].sort_values('date').reset_index(drop=True)
    
    for i, row in ad.iterrows():
        # Horizontal line from first task to last task
        ax.plot([row['first_time'], row['last_time']], [i, i], 
                color=color, linewidth=2, solid_capstyle='round')
        # Markers for first and last
        ax.scatter([row['first_time']], [i], color=color, s=20, zorder=5)
        ax.scatter([row['last_time']], [i], color=color, s=20, zorder=5)
    
    # Reference lines for 08:00 and 19:00
    ax.axvline(8, color='green', linestyle='--', alpha=0.4, linewidth=1)
    ax.axvline(19, color='red', linestyle='--', alpha=0.4, linewidth=1)
    
    ax.set_ylabel('Day index')
    ax.set_title(agent.replace('-000', '-'), fontsize=11, fontweight='bold', loc='left')
    ax.set_xlim(7, 20)
    ax.set_ylim(-1, len(ad))
    ax.grid(True, axis='x', alpha=0.3)
    
    # Y-axis: show a few date labels
    n_days = len(ad)
    if n_days > 10:
        step = max(1, n_days // 6)
        tick_idx = list(range(0, n_days, step))
        ax.set_yticks(tick_idx)
        ax.set_yticklabels([str(ad.loc[i, 'date']) for i in tick_idx], fontsize=7)
    else:
        ax.set_yticks(range(n_days))
        ax.set_yticklabels([str(d) for d in ad['date']], fontsize=7)

# X-axis formatting
axes[-1].set_xlabel('Time of Day')
axes[-1].set_xticks(range(7, 21))
axes[-1].set_xticklabels([f'{h:02d}:00' for h in range(7, 21)], fontsize=9)

fig.suptitle('Daily Work Window per Agent (green = 08:00, red = 19:00)', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Debug: check what the actual latest timestamps look like for one agent
print("\nSanity check - Technician-000001 latest timestamps per day (last 5 days):")
tech1 = df[df['resource'] == 'Technician-000001'].copy()
for d in sorted(tech1['date'].unique())[-5:]:
    day_df = tech1[tech1['date'] == d]
    last_start = day_df['start_timestamp'].max()
    last_end = day_df['end_timestamp'].max()
    print(f"  {d}: last start={last_start.strftime('%H:%M:%S')}, last end={last_end.strftime('%H:%M:%S')}, tasks={len(day_df)}")

# Summary stats
print("\nWork window summary per agent:")
print(f"{'Agent':<25} {'Earliest start':>15} {'Latest end':>12} {'Median span':>12}")
print("-" * 67)
for agent in agents:
    ad = agent_day[agent_day['resource'] == agent]
    span = ad['last_time'] - ad['first_time']
    print(f"{agent:<25} "
          f"{ad['first_time'].min():5.1f}h (med {ad['first_time'].median():.1f}h)  "
          f"{ad['last_time'].max():5.1f}h (med {ad['last_time'].median():.1f}h)  "
          f"{span.median():5.1f}h")

## 3c. Cross-Day Tasks Analysis
Do any tasks span across calendar days (start on one day, finish on another)?

In [ ]:
# Check: are there tasks where start and end fall on different calendar days?
df['start_date'] = df['start_timestamp'].dt.date
df['end_date'] = df['end_timestamp'].dt.date
cross_day = df[df['start_date'] != df['end_date']]

print(f"Cross-day tasks: {len(cross_day)} out of {len(df):,} total ({len(cross_day)/len(df)*100:.2f}%)")

if len(cross_day) > 0:
    print(f"\n--- Cross-day task details ---")
    print(f"Activities involved: {cross_day['activity_name'].unique().tolist()}")
    print(f"Agents involved: {cross_day['resource'].unique().tolist()}")
    
    # Duration of cross-day tasks
    cross_day_dur = (cross_day['end_timestamp'] - cross_day['start_timestamp'])
    print(f"\nDuration stats:")
    print(f"  Min:    {cross_day_dur.min()}")
    print(f"  Median: {cross_day_dur.median()}")
    print(f"  Max:    {cross_day_dur.max()}")
    
    # Show some examples
    print(f"\nSample cross-day tasks:")
    cols = ['activity_name', 'resource', 'start_timestamp', 'end_timestamp', 'case_id']
    display(cross_day[cols].head(10))
    
    # How many calendar days do they span?
    cross_day_span = (pd.to_datetime(cross_day['end_date']) - pd.to_datetime(cross_day['start_date'])).dt.days
    print(f"\nCalendar days spanned:")
    print(cross_day_span.value_counts().sort_index().to_string())
else:
    print("\nNo tasks span multiple calendar days.")
    print("All tasks start and finish within the same working day (08:00-19:00).")

## 4. Weekday x Hour Heatmap

In [ ]:
weekday_hour = df.groupby(['weekday', 'start_hour']).size().unstack(fill_value=0)

# Reorder weekdays
weekday_order_present = [d for d in weekday_order if d in weekday_hour.index]
weekday_hour = weekday_hour.reindex(weekday_order_present)

# Ensure all 24 hours
for h in range(24):
    if h not in weekday_hour.columns:
        weekday_hour[h] = 0
weekday_hour = weekday_hour[sorted(weekday_hour.columns)]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    weekday_hour, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
    linewidths=0.5, linecolor='white',
    xticklabels=[f'{h:02d}' for h in range(24)],
    cbar_kws={'label': 'Task Count'}
)
ax.set_title('Task Count per Weekday per Hour', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

## 5. Task Duration per Start Hour

In [ ]:
# Filter to working hours and reasonable durations
df_work = df[(df['start_hour'] >= 8) & (df['start_hour'] <= 19)].copy()
df_work = df_work[df_work['duration_minutes'] > 0]

# Cap at P95 for visualization
p95 = df_work['duration_minutes'].quantile(0.95)

fig, ax = plt.subplots(figsize=(12, 5))
df_plot = df_work[df_work['duration_minutes'] <= p95]
sns.boxplot(data=df_plot, x='start_hour', y='duration_minutes', ax=ax, 
            palette='Set2', fliersize=1)

ax.set_title(f'Task Duration by Start Hour (capped at P95 = {p95:.1f} min)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Start Hour')
ax.set_ylabel('Duration (minutes)')

plt.tight_layout()
plt.show()

# Check end-of-day effect: do 19:00 tasks have different durations?
print("\nMedian duration per start hour:")
print(df_work.groupby('start_hour')['duration_minutes'].median().round(2).to_string())

## 6. Tasks per Day Timeline

In [ ]:
# Tasks per calendar day (including zero-task days)
date_range = pd.date_range(df['start_timestamp'].min().date(), df['start_timestamp'].max().date())
daily_counts = df.groupby('date').size()
daily_full = pd.Series(0, index=[d.date() for d in date_range])
daily_full.update(daily_counts)

# Color weekends differently
colors = ['#E57373' if pd.Timestamp(d).dayofweek >= 5 else '#4CAF50' for d in daily_full.index]

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(daily_full)), daily_full.values, color=colors, width=1.0, edgecolor='white')

ax.set_title('Tasks per Day (red = weekend)', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Tasks')
ax.set_xlabel('Date')

# X-axis: show every Monday
tick_positions = []
tick_labels = []
for i, d in enumerate(daily_full.index):
    if pd.Timestamp(d).dayofweek == 0:  # Monday
        tick_positions.append(i)
        tick_labels.append(str(d))
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

weekday_days = sum(1 for d in daily_full.index if pd.Timestamp(d).dayofweek < 5)
weekend_days = sum(1 for d in daily_full.index if pd.Timestamp(d).dayofweek >= 5)
print(f"Total calendar days: {len(daily_full)}")
print(f"Weekdays: {weekday_days}, Weekends: {weekend_days}")
print(f"Weekdays with tasks: {sum(1 for d in daily_full.index if daily_full[d] > 0 and pd.Timestamp(d).dayofweek < 5)}")
print(f"Weekend days with tasks: {sum(1 for d in daily_full.index if daily_full[d] > 0 and pd.Timestamp(d).dayofweek >= 5)}")

## 7. Per-Agent Daily Activity

In [ ]:
agents = sorted(df['resource'].unique())
fig, axes = plt.subplots(len(agents), 1, figsize=(16, 3 * len(agents)), sharex=True)

for ax, agent in zip(axes, agents):
    agent_df = df[df['resource'] == agent]
    agent_daily = agent_df.groupby('date').size()
    agent_full = pd.Series(0, index=[d.date() for d in date_range])
    agent_full.update(agent_daily)
    
    colors = ['#E57373' if pd.Timestamp(d).dayofweek >= 5 else '#2196F3' for d in agent_full.index]
    ax.bar(range(len(agent_full)), agent_full.values, color=colors, width=1.0, edgecolor='white')
    ax.set_ylabel('Tasks')
    ax.set_title(agent, fontsize=11, fontweight='bold', loc='left')
    ax.set_ylim(0, agent_full.max() * 1.2 if agent_full.max() > 0 else 10)

# X-axis on bottom subplot
axes[-1].set_xticks(tick_positions)
axes[-1].set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
axes[-1].set_xlabel('Date')

fig.suptitle('Daily Task Count per Agent (red = weekend)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Conclusion

### Working Hours Profile
- **Days**: Monday-Friday only (no weekends)
- **Hours**: 08:00-19:00 (12-hour window)
- **Night work**: None (no tasks before 08:00 or after 19:00)
- **Per agent**: All agents follow identical work patterns

### Implications for Simulator
The current simulator operates 24/7 with no work schedule. This means:
1. **Throughput times are not comparable** with the original data
2. **Case durations appear artificially short** because agents work through nights/weekends
3. **The collab setting is most affected** because longer queue times push work into nights

### Recommendation
For fair comparison with the original data, the simulator should implement a work schedule:
- Working hours: **Mon-Fri, 08:00-19:00**
- When a task would complete outside working hours, defer to next business day at 08:00
- Case arrivals already follow this pattern (from original timestamps)